In [ ]:
import importlib
if importlib.util.find_spec("spytial") is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spytial-diagramming"])
from spytial import *
from spytial.annotations import *

# Bubble sort: step-by-step

Bubble sort is the cleanest possible "compare and swap" loop: scan the
array, swap adjacent out-of-order pairs, repeat until no swaps happen.
Each frame here corresponds to exactly one swap.

Like the array-backed max-heap, the spatial position of each cell is
pinned by its array index, and the algorithm forces values to migrate
between positions. Stability would fight that; `change_emphasis`
embraces it and lights up the two adjacent cells that just traded.

(We pick values that don’t overlap with the index range to keep the
visible atoms unambiguous.)

In [ ]:
from typing import List

# Cells lie left-to-right in array order.
ARRAY_VALUES = "(int.(list.idx))"
ARRAY_NEXT = "(~(list.idx)).({ i, j : int | j = add[i, 1] }).(list.idx)"

@hideAtom(selector=f'SortArray + list + (int - {ARRAY_VALUES})')
@orientation(selector=ARRAY_NEXT, directions=["right"])
@inferredEdge(selector=ARRAY_NEXT, name="")
class SortArray:
    """An array wrapper that records a frame after every swap."""

    def __init__(self, data: List[int], seq=None):
        self.a: List[int] = list(data)
        self._seq = seq
        self._snap("initial array")

    def _snap(self, label):
        if self._seq is not None:
            self._seq.record(self, label=label)

    def bubble_sort(self):
        n = len(self.a)
        for i in range(n - 1):
            swapped = False
            for j in range(n - 1 - i):
                if self.a[j] > self.a[j + 1]:
                    self.a[j], self.a[j + 1] = self.a[j + 1], self.a[j]
                    self._snap(f"swap a[{j}] <-> a[{j+1}]")
                    swapped = True
            if not swapped:
                break

    def __repr__(self):
        return f"SortArray({self.a})"

## Demo

Sort a small array. Each frame is exactly one adjacent-swap.

In [ ]:
seq = sequence(
    sequence_policy="change_emphasis",
    title="Bubble sort: one swap per frame",
)

# Values chosen to be disjoint from index range [0..6] so the visible atoms are unambiguous.
arr = SortArray([29, 10, 14, 37, 13, 23, 11], seq=seq)
arr.bubble_sort()

seq.diagram(method="browser")